# SEDI Index

In [ ]:
# general use packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# packages for altering time to match up!
import sys
import cftime

# climpred packages
import climpred
from climpred import HindcastEnsemble
from climpred.tutorial import load_dataset
from climpred.stats import rm_poly

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat

import scipy.stats as stats

In [ ]:
import sklearn.metrics as sk

In [ ]:
import xskillscore

## Calculate

In [ ]:
var = 'dRHO'
var2 = 'dRHO'
depth = 'surface'

In [ ]:
# smyle02 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var + '.monthly.' + depth + '.2.regrid.nc')[var]
# smyle02_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.2.time.nc')

# smyle05 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var + '.monthly.' + depth + '.5.regrid.nc')[var]
# smyle05_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.5.time.nc')

# smyle08 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var + '.monthly.' + depth + '.8.regrid.nc')[var]
# smyle08_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.8.time.nc')

smyle11 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var + '.monthly.' + depth + '.11.regrid.nc')[var]
smyle11_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.11.time.nc')

In [ ]:
%%time
# remove climatological drift from the data
# smyle02_anom,smyle02_clim = stat.remove_drift(smyle02,smyle02_time,1972,2017)
# smyle05_anom,smyle05_clim = stat.remove_drift(smyle05,smyle05_time,1972,2017)
# smyle08_anom,smyle08_clim = stat.remove_drift(smyle08,smyle08_time,1972,2017)
smyle11_anom,smyle11_clim = stat.remove_drift(smyle11,smyle11_time,1972,2017)

In [ ]:
%%time
# select the anomaly!

# smyle02_anom = smyle02_anom.time
# smyle05_anom = smyle05_anom.time
# smyle08_anom = smyle08_anom.time
smyle11_anom = smyle11_anom.time

In [ ]:
obs = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/LR/dRHO.regrid.nc')['dRHO']
obs['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
obs = obs.groupby('time.month') - obs.groupby('time.month').mean()

### mask land as NaNs

In [ ]:
ds = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/OceanSODA-ETHZ_GRaCER_v2021a_1982-2020.nc')['temperature'].isel(time=0).drop('time')

ds = ds.where(ds.lat < 65)

In [ ]:
# obs = obs.sel(lat = slice(-78,90))
obs = obs.where(np.isnan(ds) == 0, np.NaN)

### Calculate ACC

In [ ]:
def leadtime_corr_byseas(mod_da,mod_time,obs_da,detrend=False):
    """ 
    Computes the correlation coefficient between two xarray DataArrays, which 
    must share the same lat/lon coordinates (if any). Assumes time coordinates are roughly compatible
    between model and obs.
    
        Inputs
        mod_da: a seasonally-averaged hindcast DataArray dimensioned (Y,L,M,...)
        obs_da: an OBS DataArray dimensioned (time,...)
        mod_time: a hindcast time DataArray dimensioned (Y,L). NOTE: assumes mod_time.dt.month
            returns mid-month of 3-month seasonal average (e.g., mon=1 ==> "DJF").
    """
    mod_da = mod_da.mean('M')
    r_list = []   
    r_alt_list = []
    p_list = []

    damped_persist = autocorr(mod_da,mod_time,obs_da,detrend=True)
    
    # use these for calculations of skill!
    for i in range(1,14): #mod_da.L.values:
        ens = mod_da.sel(L=i).rename({'Y':'time'})
        ens_time_year = mod_time.sel(L=i).time.dt.year.data
        ens_time_month = mod_time.sel(L=i).time.dt.month.data[0]
        ens_ts = ens.assign_coords(time=("time",ens_time_year))
        
        obs_ts = obs_da.where(obs_da.time.dt.month == ens_time_month,drop=True)
        
        obs_ts['time'] = obs_ts['time'].dt.year
    # return ens_ts, obs_ts
        a,b = xr.align(ens_ts,obs_ts)
        
        # if detrend:
        #         a = detrend_linear(a,'time')
        #         b = detrend_linear(b,'time')
                
        init_acc = xr.corr(a,b,dim='time')
        r_list.append(init_acc)
    
    corr = xr.concat(r_list,mod_da.L)
    # corr_alt = xr.concat(r_alt_list,mod_da.L)
    return xr.Dataset({'corr':corr}), xr.Dataset({'p':p}),

In [ ]:
%%time
# a = leadtime_corr_byseas(smyle02_anom.isel(L=slice(0,13)),smyle02_time,obs,detrend=False)
# a = leadtime_corr_byseas(smyle05_anom.isel(L=slice(0,13)),smyle05_time,obs,detrend=False)
# a = leadtime_corr_byseas(smyle08_anom.isel(L=slice(0,13)),smyle08_time,obs,detrend=False)
a = leadtime_corr_byseas(smyle11_anom.isel(L=slice(0,13)),smyle11_time,obs,detrend=False)

In [ ]:
# final.to_netcdf('./ACC/dRHO.ACC.08.p-value.autocorr.nc')
a[2].to_netcdf('./ACC/dRHO.ACC.11.damped_persist.nc')

## Plot

### combine them

In [ ]:
dRHO_02 = xr.open_dataset('./ACC/dRHO.ACC.02.nc')['corr']
dRHO_05 = xr.open_dataset('./ACC/dRHO.ACC.05.nc')['corr']
dRHO_08 = xr.open_dataset('./ACC/dRHO.ACC.08.nc')['corr']
dRHO_11 = xr.open_dataset('./ACC/dRHO.ACC.11.nc')['corr']

dRHO_02 = xr.open_dataset('./ACC/dRHO.ACC.02.p-value.nc')#['corr']
dRHO_05 = xr.open_dataset('./ACC/dRHO.ACC.05.p-value.nc')#['corr']
dRHO_08 = xr.open_dataset('./ACC/dRHO.ACC.08.p-value.nc')#['corr']
dRHO_11 = xr.open_dataset('./ACC/dRHO.ACC.11.p-value.nc')#['corr']

In [ ]:
dRHO_02_sig = dRHO_02.corr.where(dRHO_02.p < 0.05)
dRHO_05_sig = dRHO_05.corr.where(dRHO_05.p < 0.05)
dRHO_08_sig = dRHO_08.corr.where(dRHO_08.p < 0.05)
dRHO_11_sig = dRHO_11.corr.where(dRHO_11.p < 0.05)

In [ ]:
# FISHER-Z TRANSFORM
dRHO_02_trans_sig = np.arctan(dRHO_02_sig)
dRHO_05_trans_sig = np.arctan(dRHO_05_sig)
dRHO_08_trans_sig = np.arctan(dRHO_08_sig)
dRHO_11_trans_sig = np.arctan(dRHO_11_sig)

avg_sig = (dRHO_02_trans_sig + dRHO_05_trans_sig + dRHO_08_trans_sig + dRHO_11_trans_sig)/4

final_mean_sig = np.tan(avg_sig)

In [ ]:
final_mean_sig['L'] = final_mean_sig['L']+ 1

In [ ]:
final_mean.to_netcdf('./ACC/combined.average.nc')
final_mean_sig.to_netcdf('./ACC/combined.average.sig.nc')

### F3

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches

In [ ]:
from cartopy.util import add_cyclic_point
def xr_add_cyclic_point(da):
    """
    Inputs
    da: xr.DataArray with dimensions (time,lat,lon)
    """

    # Use add_cyclic_point to interpolate input data
    lon_idx = da.dims.index('lon')
    wrap_data, wrap_lon = add_cyclic_point(da.values, coord=da.lon, axis=lon_idx)

    # Generate output DataArray with new data but same structure as input
    outp_da = xr.DataArray(data=wrap_data, 
                           coords = {'L': da.L, 'lat': da.lat, 'lon': wrap_lon}, 
                           dims=da.dims, 
                           attrs=da.attrs)
    
    return outp_da

In [ ]:
sea_ice = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/LR/IFRAC.regrid.nc')['IFRAC']
sea_ice = sea_ice.mean('time')
ice_mask = (np.isnan(sea_ice.where(sea_ice > 0)))

In [ ]:
final_mean = final_mean.where(ice_mask)
sig = final_mean_sig.where(ice_mask)

In [ ]:
final_mean = xr_add_cyclic_point(final_mean)
sig = xr_add_cyclic_point(sig)
# ice_mask = xr_add_cyclic_point(ice_mask)

In [ ]:
months = [0,1,2,3,4,5]
f, ax = plt.subplots(6,1,figsize=(3,8),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

for i in range(6):
    im = final_mean.isel(L = [i]).squeeze().plot.contourf(ax = ax[i],levels=np.arange(-1,1.05,0.05),cmap='PRGn',transform = ccrs.PlateCarree(),add_colorbar = False)
    
    (np.isnan(sig).isel(L=[i]).where(np.isnan(sig).isel(L=[i]) == 1)).squeeze().where(ice_mask).plot.contourf(ax=ax[i],transform = ccrs.PlateCarree(),alpha=0,hatches=['....'],add_colorbar=False)

    # extreme_base.isel(L=i).plot.contour(ax = ax[i], levels=[0.5],cmap='k',zorder=100,transform = ccrs.PlateCarree(),add_colorbar = False,linewidths=0.5)
    ax[i].add_feature(cfeature.LAND,color='k',zorder=10)
    # ax[i,0].set_title(str(months[i] + 1) + ' month lead')
    ax[i].add_patch(mpatches.Rectangle(xy=[-125, -5], width=40, height=10, facecolor='none', edgecolor='k', transform=ccrs.PlateCarree()))
    ax[i].add_patch(mpatches.Rectangle(xy=[-180, -5], width=40, height=10, facecolor='none', edgecolor='k', transform=ccrs.PlateCarree()))
    ax[i].add_patch(mpatches.Rectangle(xy=[-70, 15], width=40, height=10, facecolor='none', edgecolor='k', transform=ccrs.PlateCarree()))
    ax[i].set_title('')
    ax[i].set_global()

plt.subplots_adjust(wspace=0.1, hspace=0.1)

f.savefig('./figures/clean/dRHO.predictability.boxes.new_colorbar.sig.png',dpi=500)


In [ ]:
f, ax = plt.subplots(1,1,subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))
# f.subplots_adjust(right=0.8)
cbar_ax = f.add_axes([0.88, 0.3, 0.07, 0.80])
cbar = f.colorbar(im, cax=cbar_ax, ticks=[0])
cbar.set_ticks([-1, 0, 1])
cbar.set_ticklabels(['-1', '0', '1'])
# cbar.ax.tick_params(labelsize=10,size=[])
cbar.set_label('', rotation=270,fontsize=14)
f.savefig('./figures/clean/dRHO.predictability.boxes.new_colorbar.COLORBAR.pdf')
